In [ ]:
#需要使用L4GPU，然后官方代码的专家数，从64改为16
#把依赖文件kernel.py上传
#注意本地没有GPU没法运行这个代码，要用colab或者autodl，torch版本太低不可以，因为没有rms归一化，还有本地没安装triton不可以

In [1]:
!pip list|grep triton
# 要求结果是triton 3.1.0


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!ls

sample_data


# 下面是kernel.py的代码，直接复制进来了，为了避免每次上传kernel.py

In [ ]:
from typing import Tuple  # 导入Tuple类型提示 / Import Tuple type hint

import torch  # 导入PyTorch库 / Import PyTorch library
import triton  # 导入Triton库用于GPU加速 / Import Triton library for GPU acceleration
import triton.language as tl  # 导入Triton语言模块作为tl / Import Triton language module as tl
from triton import Config  # 从Triton导入Config类 / Import Config class from Triton


@triton.jit  # Triton JIT装饰器,用于编译为GPU代码 / Triton JIT decorator for GPU code compilation
def act_quant_kernel(x_ptr, y_ptr, s_ptr, BLOCK_SIZE: tl.constexpr):
    """
    量化输入张量x_ptr并将结果存储在y_ptr中,将缩放因子存储在s_ptr中
    Quantizes the input tensor `x_ptr` and stores the result in `y_ptr` and the scaling factor in `s_ptr`

    参数 / Args:
        x_ptr (triton.Pointer): 输入张量的指针 / Pointer to input tensor
        y_ptr (triton.Pointer): 输出张量的指针,用于存储量化值 / Pointer to output tensor for quantized values
        s_ptr (triton.Pointer): 输出张量的指针,用于存储缩放因子 / Pointer to output tensor for scaling factors
        BLOCK_SIZE (tl.constexpr): 每个程序实例处理的块大小 / Block size processed by each program instance

    返回 / Returns:
        None
    """
    pid = tl.program_id(axis=0)  # 获取程序ID / Get program ID
    offs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)  # 计算偏移量 / Calculate offsets
    x = tl.load(x_ptr + offs).to(tl.float32)  # 加载输入数据并转换为float32 / Load input data and convert to float32
    s = tl.max(tl.abs(x)) / 448.  # 计算缩放因子 / Calculate scaling factor
    y = x / s  # 应用缩放 / Apply scaling
    y = y.to(y_ptr.dtype.element_ty)  # 转换为目标数据类型 / Convert to target dtype
    tl.store(y_ptr + offs, y)  # 存储量化结果 / Store quantized results
    tl.store(s_ptr + pid, s)  # 存储缩放因子 / Store scaling factor


def act_quant(x: torch.Tensor, block_size: int = 128) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    使用分块量化对输入张量x进行量化
    Quantizes the input tensor `x` using block-wise quantization

    参数 / Args:
        x (torch.Tensor): 待量化的输入张量,必须是连续的且最后一维大小必须能被block_size整除 
                         Input tensor to be quantized, must be contiguous and its last dimension size must be divisible by block_size
        block_size (int, optional): 用于量化的块大小,默认为128 / Block size for quantization, default is 128

    返回 / Returns:
        Tuple[torch.Tensor, torch.Tensor]: 包含以下内容的元组 / A tuple containing:
            - 量化后的张量,dtype为torch.float8_e4m3fn / Quantized tensor with dtype torch.float8_e4m3fn
            - 缩放因子张量,dtype为torch.float32 / Scaling factors tensor with dtype torch.float32
    """
    assert x.is_contiguous(), '输入张量必须是连续的 / Input tensor must be contiguous'
    assert x.size(-1) % block_size == 0, f'最后维度大小必须能被block_size整除 (block_size={block_size}) / Last dimension size must be divisible by block_size'
    y = torch.empty_like(x, dtype=torch.float8_e4m3fn)  # 创建输出张量 / Create output tensor
    s = x.new_empty(*x.size()[:-1], x.size(-1) // block_size, dtype=torch.float32)  # 创建缩放因子张量 / Create scaling factors tensor
    grid = lambda meta: (triton.cdiv(x.numel(), meta['BLOCK_SIZE']), )  # 定义网格大小 / Define grid size
    act_quant_kernel[grid](x, y, s, BLOCK_SIZE=block_size)  # 执行量化kernel / Execute quantization kernel
    return y, s


@triton.jit  # Triton JIT装饰器 / Triton JIT decorator
def weight_dequant_kernel(x_ptr, s_ptr, y_ptr, M, N, BLOCK_SIZE: tl.constexpr):
    """
    使用提供的缩放因子对权重进行反量化并存储结果
    Dequantizes weights using provided scaling factors and stores the result

    参数 / Args:
        x_ptr (tl.pointer): 量化权重的指针 / Pointer to quantized weights
        s_ptr (tl.pointer): 缩放因子的指针 / Pointer to scaling factors
        y_ptr (tl.pointer): 反量化权重的输出缓冲区指针 / Pointer to output buffer for dequantized weights
        M (int): 权重矩阵的行数 / Number of rows in weight matrix
        N (int): 权重矩阵的列数 / Number of columns in weight matrix
        BLOCK_SIZE (tl.constexpr): 分块大小 / Block size for tiling

    返回 / Returns:
        None
    """
    pid_m = tl.program_id(axis=0)  # 获取m维度的程序ID / Get program ID for m dimension
    pid_n = tl.program_id(axis=1)  # 获取n维度的程序ID / Get program ID for n dimension
    n = tl.cdiv(N, BLOCK_SIZE)  # 计算n方向的块数 / Calculate number of blocks in n direction
    offs_m = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)  # 计算m方向的偏移量 / Calculate offsets in m direction
    offs_n = pid_n * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)  # 计算n方向的偏移量 / Calculate offsets in n direction
    offs = offs_m[:, None] * N + offs_n[None, :]  # 计算总偏移量 / Calculate total offsets
    mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)  # 创建掩码 / Create mask
    x = tl.load(x_ptr + offs, mask=mask).to(tl.float32)  # 加载量化数据 / Load quantized data
    s = tl.load(s_ptr + pid_m * n + pid_n)  # 加载缩放因子 / Load scaling factor
    y = x * s  # 应用缩放 / Apply scaling
    tl.store(y_ptr + offs, y, mask=mask)  # 存储结果 / Store results


def weight_dequant(x: torch.Tensor, s: torch.Tensor, block_size: int = 128) -> torch.Tensor:
    """
    使用提供的缩放张量对给定的权重张量进行反量化
    Dequantizes given weight tensor using provided scale tensor

    参数 / Args:
        x (torch.Tensor): 形状为(M, N)的量化权重张量 / Quantized weight tensor of shape (M, N)
        s (torch.Tensor): 形状为(M, N)的缩放张量 / Scale tensor of shape (M, N)
        block_size (int, optional): 用于反量化的块大小,默认为128 / Block size for dequantization, defaults to 128

    返回 / Returns:
        torch.Tensor: 与x形状相同的反量化权重张量 / Dequantized weight tensor of same shape as x

    异常 / Raises:
        AssertionError: 如果x或s不是连续的或维度不是2 / If x or s are not contiguous or dimensions are not 2
    """
    assert x.is_contiguous() and s.is_contiguous(), '输入张量必须是连续的 / Input tensors must be contiguous'
    assert x.dim() == 2 and s.dim() == 2, '输入张量必须是2维的 / Input tensors must have 2 dimensions'
    M, N = x.size()  # 获取矩阵维度 / Get matrix dimensions
    y = torch.empty_like(x, dtype=torch.get_default_dtype())  # 创建输出张量 / Create output tensor
    grid = lambda meta: (triton.cdiv(M, meta['BLOCK_SIZE']), triton.cdiv(N, meta['BLOCK_SIZE']))  # 定义网格大小 / Define grid size
    weight_dequant_kernel[grid](x, s, y, M, N, BLOCK_SIZE=block_size)  # 执行反量化kernel / Execute dequantization kernel
    return y


fp8_gemm_configs = [  # FP8 GEMM配置列表 / List of FP8 GEMM configurations
    Config({'BLOCK_SIZE_M': block_m, 'BLOCK_SIZE_N': block_n, 'BLOCK_SIZE_K': 128}, num_stages=num_stages, num_warps=8)
    for block_m in [16, 32, 64] for block_n in [32, 64, 128] for num_stages in [3, 4, 5, 6]
]

@triton.autotune(configs=fp8_gemm_configs, key=['N', 'K'])  # Triton自动调优装饰器 / Triton autotune decorator
@triton.jit  # Triton JIT装饰器 / Triton JIT decorator
def fp8_gemm_kernel(a_ptr, b_ptr, c_ptr,
                    a_s_ptr, b_s_ptr,
                    M, N: tl.constexpr, K: tl.constexpr,
                    BLOCK_SIZE_M: tl.constexpr,
                    BLOCK_SIZE_N: tl.constexpr,
                    BLOCK_SIZE_K: tl.constexpr):
    """
    在FP8矩阵上执行矩阵乘法运算,包含缩放因子
    Performs matrix multiplication on FP8 matrices with scaling factors

    参数 / Args:
        a_ptr (tl.tensor): 第一个输入矩阵A的指针 / Pointer to first input matrix A
        b_ptr (tl.tensor): 第二个输入矩阵B的指针 / Pointer to second input matrix B
        c_ptr (tl.tensor): 输出矩阵C的指针 / Pointer to output matrix C
        a_s_ptr (tl.tensor): 矩阵A的缩放因子指针 / Pointer to scaling factors for matrix A
        b_s_ptr (tl.tensor): 矩阵B的缩放因子指针 / Pointer to scaling factors for matrix B
        M (int): 矩阵A和C的行数 / Number of rows in matrix A and C
        N (tl.constexpr): 矩阵B和C的列数 / Number of columns in matrix B and C
        K (tl.constexpr): 矩阵A的列数和矩阵B的行数 / Number of columns in A and rows in B
        BLOCK_SIZE_M/N/K (tl.constexpr): M/N/K维度的块大小 / Block sizes for M/N/K dimensions

    返回 / Returns:
        None
    """
    pid_m = tl.program_id(axis=0)  # 获取m维度的程序ID / Get program ID for m dimension
    pid_n = tl.program_id(axis=1)  # 获取n维度的程序ID / Get program ID for n dimension
    k = tl.cdiv(K, BLOCK_SIZE_K)  # 计算K维度的块数 / Calculate number of blocks in K dimension
    offs_m = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M  # 计算M维度的偏移量 / Calculate offsets for M dimension
    offs_n = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N  # 计算N维度的偏移量 / Calculate offsets for N dimension
    offs_k = tl.arange(0, BLOCK_SIZE_K)  # 计算K维度的偏移量 / Calculate offsets for K dimension
    a_ptrs = a_ptr + offs_m[:, None] * K + offs_k[None, :]  # 计算矩阵A的指针 / Calculate pointers for matrix A
    b_ptrs = b_ptr + offs_n[None, :] * K + offs_k[:, None]  # 计算矩阵B的指针 / Calculate pointers for matrix B
    a_s_ptrs = a_s_ptr + offs_m * k  # 计算矩阵A缩放因子的指针 / Calculate pointers for matrix A scaling factors
    b_s_ptrs = b_s_ptr + (offs_n // BLOCK_SIZE_K) * k  # 计算矩阵B缩放因子的指针 / Calculate pointers for matrix B scaling factors

    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)  # 初始化累加器 / Initialize accumulator
    for i in range(k):  # 对每个块进行循环 / Loop over blocks
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - i * BLOCK_SIZE_K, other=0.0)  # 加载矩阵A的数据 / Load data from matrix A
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - i * BLOCK_SIZE_K, other=0.0)  # 加载矩阵B的数据 / Load data from matrix B
        a_s = tl.load(a_s_ptrs)  # 加载矩阵A的缩放因子 / Load scaling factors for matrix A
        b_s = tl.load(b_s_ptrs)  # 加载矩阵B的缩放因子 / Load scaling factors for matrix B
        accumulator += tl.dot(a, b) * a_s[:, None] * b_s[None, :]  # 执行矩阵乘法并应用缩放 / Perform matrix multiplication and apply scaling
        a_ptrs += BLOCK_SIZE_K  # 更新矩阵A的指针 / Update pointers for matrix A
        b_ptrs += BLOCK_SIZE_K  # 更新矩阵B的指针 / Update pointers for matrix B
        a_s_ptrs += 1  # 更新矩阵A缩放因子的指针 / Update pointers for matrix A scaling factors
        b_s_ptrs += 1  # 更新矩阵B缩放因子的指针 / Update pointers for matrix B scaling factors
    c = accumulator.to(c_ptr.dtype.element_ty)  # 转换累加器类型 / Convert accumulator type
    offs_m = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)  # 计算输出矩阵M维度的偏移量 / Calculate output matrix offsets for M dimension
    offs_n = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)  # 计算输出矩阵N维度的偏移量 / Calculate output matrix offsets for N dimension
    c_ptrs = c_ptr + offs_m[:, None] * N + offs_n[None, :]  # 计算输出矩阵的指针 / Calculate pointers for output matrix
    mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)  # 创建掩码 / Create mask
    tl.store(c_ptrs, c, mask=mask)  # 存储结果 / Store results


def fp8_gemm(a: torch.Tensor, a_s: torch.Tensor, b: torch.Tensor, b_s: torch.Tensor):
    """
    使用FP8精度执行矩阵乘法
    Perform matrix multiplication using FP8 precision

    参数 / Args:
        a (torch.Tensor): 第一个输入矩阵,必须是连续的 / First input matrix, must be contiguous
        a_s (torch.Tensor): 第一个输入矩阵的缩放因子,必须是连续的 / Scaling factor for first input matrix, must be contiguous
        b (torch.Tensor): 第二个输入矩阵,必须是连续的 / Second input matrix, must be contiguous
        b_s (torch.Tensor): 第二个输入矩阵的缩放因子,必须是连续的 / Scaling factor for second input matrix, must be contiguous

    返回 / Returns:
        torch.Tensor: 矩阵乘法的结果 / Result of matrix multiplication
    """
    assert a.is_contiguous() and b.is_contiguous(), '输入张量必须是连续的 / Input tensors must be contiguous'
    assert a_s.is_contiguous() and b_s.is_contiguous(), '缩放因子张量必须是连续的 / Scaling factor tensors must be contiguous'
    K = a.size(-1)  # 获取K维度大小 / Get size of K dimension
    M = a.numel() // K  # 计算M维度大小 / Calculate size of M dimension
    N = b.size(0)  # 获取N维度大小 / Get size of N dimension
    c = a.new_empty(*a.size()[:-1], N, dtype=torch.get_default_dtype())  # 创建输出张量 / Create output tensor
    grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']), triton.cdiv(N, META['BLOCK_SIZE_N']))  # 定义网格大小 / Define grid size
    fp8_gemm_kernel[grid](a, b, c, a_s, b_s, M, N, K)  # 执行FP8 GEMM kernel / Execute FP8 GEMM kernel
    return c


In [ ]:
import math  # 导入数学运算模块 / Import math module for mathematical operations
from dataclasses import dataclass  # 导入数据类装饰器 / Import dataclass decorator for creating data classes
from typing import Tuple, Optional, Literal  # 导入类型提示工具 / Import typing hints

import torch  # 导入PyTorch深度学习框架 / Import PyTorch deep learning framework
from torch import nn  # 导入神经网络模块 / Import neural network module
import torch.nn.functional as F  # 导入神经网络函数库 / Import neural network functional library
import torch.distributed as dist  # 导入分布式训练模块 / Import distributed training module


print(torch.__version__)  # 打印PyTorch版本号 / Print PyTorch version


# 定义全局变量
world_size = 1  # 世界大小（用于分布式计算）
rank = 0  # 当前进程的排名
block_size = 128  # 块大小（可能用于矩阵运算优化）
gemm_impl: Literal["bf16", "fp8"] = "bf16"  # 矩阵乘法实现方式（bf16 或 fp8）
attn_impl: Literal["naive", "absorb"] = "naive"  # 注意力机制实现方式

@dataclass  # 使用dataclass装饰器创建数据类 / Use dataclass decorator to create a data class
class ModelArgs:
    """
    用于定义模型参数和超参数的数据类。

    属性：
        max_batch_size (int): 最大批量大小。
        max_seq_len (int): 最大序列长度。
        dtype (Literal["bf16", "fp8"]): 计算数据类型（bf16 或 fp8）。
        vocab_size (int): 词汇表大小。
        dim (int): 模型隐藏层维度。
        inter_dim (int): MLP 层的中间层维度。
        moe_inter_dim (int): MoE 层的中间层维度。
        n_layers (int): Transformer 层的数量。
        n_dense_layers (int): 模型中的全连接层数量。
        n_heads (int): 注意力头的数量。
        n_routed_experts (int): MoE 层中可路由的专家数量。
        n_shared_experts (int): MoE 层中共享的专家数量。
        n_activated_experts (int): MoE 层中每次激活的专家数量。
        n_expert_groups (int): MoE 层中的专家组数量。
        n_limited_groups (int): MoE 路由中的受限组数量。
        score_func (Literal["softmax", "sigmoid"]): MoE 路由的评分函数。
        route_scale (float): MoE 路由评分的缩放因子。
        q_lora_rank (int): 查询（Query）投影的 LoRA 秩。
        kv_lora_rank (int): 键值（Key-Value）投影的 LoRA 秩。
        qk_nope_head_dim (int): 无位置编码的 Query-Key 投影维度。
        qk_rope_head_dim (int): 使用旋转位置编码（RoPE）的 Query-Key 投影维度。
        v_head_dim (int): 值（Value）投影维度。
        original_seq_len (int): 原始序列长度。
        rope_theta (float): 旋转位置编码的基数。
        rope_factor (float): 扩展序列长度的缩放因子。
        beta_fast (int): 快速 Beta 修正因子。
        beta_slow (int): 慢速 Beta 修正因子。
        mscale (float): 扩展注意力的缩放因子。
    """
    max_batch_size: int = 8  # 训练时的最大批次大小
    max_seq_len: int = 4096 * 4  # 允许的最大序列长度（可能用于扩展序列训练）
    dtype: Literal["bf16", "fp8"] = "bf16"  # 计算数据类型，默认使用 bfloat16
    vocab_size: int = 102400  # 词汇表大小
    dim: int = 2048  # 模型隐藏层维度
    inter_dim: int = 10944  # MLP 层的中间层维度
    moe_inter_dim: int = 1408  # MoE 层的中间层维度
    n_layers: int = 3  # Transformer 层数,原本官方这里是27
    n_dense_layers: int = 1  # 全连接层数
    n_heads: int = 16  # 注意力头数

    # MoE（Mixture of Experts）相关参数
    n_routed_experts: int = 16  # 可被路由的专家数量,论文里DeepSeekV3用了256个专家
    n_shared_experts: int = 2  # 共享专家数量，始终都激活的专家数量，来保证模型的基线性能
    n_activated_experts: int = 6  # 每次激活的专家数量
    n_expert_groups: int = 1  # 专家组数量（可能用于分组专家路由）
    n_limited_groups: int = 1  # 受限专家组数量
    score_func: Literal["softmax", "sigmoid"] = "softmax"  # MoE 路由评分函数，默认使用 softmax
    route_scale: float = 1.0  # 路由评分的缩放因子

    # MLA（Multi-Level Attention）相关参数
    q_lora_rank: int = 0  # Query 投影的 LoRA 秩（低秩适配）,为0不做低秩变换
    kv_lora_rank: int = 512  # Key-Value 投影的 LoRA 秩
    qk_nope_head_dim: int = 128  # 无 RoPE 的 Query-Key 维度
    qk_rope_head_dim: int = 64  # 使用 RoPE 的 Query-Key 维度
    v_head_dim: int = 128  # Value 维度

    # YARN（Yet Another RoPE Network）相关参数
    original_seq_len: int = 4096  # 原始序列长度
    rope_theta: float = 10000.0  # RoPE 旋转位置编码的基数
    rope_factor: float = 40  # 扩展序列长度的缩放因子
    beta_fast: int = 32  # 快速 Beta 修正因子
    beta_slow: int = 1  # 慢速 Beta 修正因子
    mscale: float = 1.0  # 扩展注意力的缩放因子

2.6.0+cu124


In [ ]:
import torch  # 导入PyTorch主包
import torch.nn as nn  # 导入神经网络模块
import torch.nn.functional as F  # 导入函数式API
import torch.distributed as dist  # 导入分布式训练模块
from typing import Optional  # 导入Optional类型提示

# 假设 world_size 和 rank 由分布式训练环境提供
world_size = dist.get_world_size() if dist.is_initialized() else 1  # 获取分布式训练的总进程数,未初始化则为1
rank = dist.get_rank() if dist.is_initialized() else 0  # 获取当前进程的rank,未初始化则为0

class ParallelEmbedding(nn.Module):
    """
    并行嵌入层（ParallelEmbedding），支持分布式训练环境下的词向量分片。

    参数:
        vocab_size (int): 词表大小，即整个模型的词汇总数。
        dim (int): 词向量的维度。

    说明:
        - 词表在多个进程（GPU）之间进行分片，每个进程仅存储词表的一部分。
        - vocab_size 必须能够被 world_size 整除，以确保各 GPU 拥有相同大小的词向量片段。
    """
    def __init__(self, vocab_size: int, dim: int):
        super().__init__()  # 调用父类初始化
        self.vocab_size = vocab_size  # 词表大小
        self.dim = dim  # 词向量维度
        assert vocab_size % world_size == 0, f"词表大小必须能被 world_size 整除 (world_size={world_size})"

        # 计算当前进程（GPU）负责的词表片段大小
        self.part_vocab_size = vocab_size // world_size  # 每个进程负责的词表大小
        # 计算当前进程的词表起始和结束索引
        self.vocab_start_idx = rank * self.part_vocab_size  # 当前进程词表的起始索引
        self.vocab_end_idx = self.vocab_start_idx + self.part_vocab_size  # 当前进程词表的结束索引
        # 初始化当前进程的词向量参数
        self.weight = nn.Parameter(torch.empty(self.part_vocab_size, self.dim))  # 创建词向量参数矩阵

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        并行嵌入层的前向传播。

        参数:
            x (torch.Tensor): 输入张量，包含 token 的索引。

        返回:
            torch.Tensor: 词向量表示。

        处理流程:
            1. 如果使用多 GPU 训练（world_size > 1），检查 token 是否属于当前 GPU 负责的词表范围:
                - 若 token 超出当前 GPU 负责的范围，设为 0（避免索引超界）。
            2. 计算词嵌入。
            3. 若是多 GPU 训练，则进行 all_reduce 操作，将所有 GPU 的嵌入求和（同步）。
        """
        if world_size > 1:  # 如果是多GPU训练环境
            # 生成掩码，标记不属于当前进程词表范围的 token
            mask = (x < self.vocab_start_idx) | (x >= self.vocab_end_idx)  # 创建超出范围的token掩码
            # 将输入索引映射到当前进程的词表范围内
            x = x - self.vocab_start_idx  # 将全局索引转换为局部索引
            # 将超出范围的索引设为 0，避免索引超界
            x[mask] = 0  # 将超出范围的token索引设为0

        # 获取嵌入
        y = F.embedding(x, self.weight)  # 执行词嵌入操作

        if world_size > 1:  # 如果是多GPU训练环境
            # 对超出范围的 token 设置为 0
            y[mask] = 0  # 将超出范围token的嵌入向量置零
            # all_reduce 操作，确保所有进程得到相同的词嵌入
            dist.all_reduce(y)  # 在所有进程间同步并累加嵌入向量

        return y  # 返回最终的词向量表示


def linear(x: torch.Tensor, weight: torch.Tensor, bias: Optional[torch.Tensor] = None) -> torch.Tensor:
    """
    线性变换函数，实现 y = xA^T + b，支持量化权重的计算。

    参数:
        x (torch.Tensor): 输入张量。
        weight (torch.Tensor): 权重张量，可能是量化后的，需要进行解量化处理。
        bias (Optional[torch.Tensor]): 偏置项（可选），默认为 None。

    返回:
        torch.Tensor: 线性变换后的结果。

    说明:
        - 若 weight 不是量化的，则直接调用 F.linear 计算。
        - 若 weight 是量化的（element_size() == 1），则需要先进行解量化，再进行计算。
        - 当 gemm_impl == "bf16" 时，使用 bf16 计算。
        - 其他情况，对 x 进行量化，然后使用 fp8_gemm 计算。
    """
    if weight.element_size() > 1:  # 如果权重不是量化的(每个元素大于1字节)
        # 直接使用标准的 F.linear 计算
        return F.linear(x, weight, bias)  # 执行标准线性变换
    elif gemm_impl == "bf16":  # 如果使用bfloat16实现
        # 量化权重，需要解量化
        weight = weight_dequant(weight, weight.scale)  # 对权重进行解量化
        return F.linear(x, weight, bias)  # 执行线性变换
    else:  # 其他情况(使用fp8实现)
        # 其他情况：对 x 进行量化，并使用 fp8_gemm 计算
        x, scale = act_quant(x, block_size)  # 对输入进行量化
        y = fp8_gemm(x, scale, weight, weight.scale)  # 使用fp8 gemm计算
        if bias is not None:  # 如果有偏置项
            y += bias  # 添加偏置
        return y  # 返回计算结果


class Linear(nn.Module):
    """
    自定义线性层，支持量化权重，并提供可选的偏置项。

    参数:
        in_features (int): 输入特征维度。
        out_features (int): 输出特征维度。
        bias (bool): 是否包含偏置项，默认为 False。
        dtype (可选): 计算数据类型，默认为 torch.bfloat16。

    说明:
        - 如果 weight 是量化的，则需要额外存储 scale 参数。
        - bias 可选，若不使用，则注册为 None。
    """
    dtype = torch.bfloat16  # 默认数据类型

    def __init__(self, in_features: int, out_features: int, bias: bool = False, dtype=None):
        super().__init__()  # 调用父类初始化
        self.in_features = in_features  # 输入特征维度
        self.out_features = out_features  # 输出特征维度
        self.weight = nn.Parameter(torch.empty(out_features, in_features, dtype=dtype or Linear.dtype))  # 创建权重参数

        # 若权重是量化的（element_size() == 1），则需要 scale 参数
        if self.weight.element_size() == 1:  # 如果权重是量化的(每个元素1字节)
            scale_out_features = (out_features + block_size - 1) // block_size  # 计算输出维度的scale大小
            scale_in_features = (in_features + block_size - 1) // block_size  # 计算输入维度的scale大小
            # 存储量化 scale 参数
            self.weight.scale = self.scale = nn.Parameter(torch.empty(scale_out_features, scale_in_features, dtype=torch.float32))  # 创建scale参数
        else:  # 如果权重不是量化的
            # 非量化情况，无需 scale 参数
            self.register_parameter("scale", None)  # 注册空的scale参数

        # 处理偏置
        if bias:  # 如果需要偏置
            self.bias = nn.Parameter(torch.empty(out_features))  # 创建偏置参数
        else:  # 如果不需要偏置
            self.register_parameter("bias", None)  # 注册空的偏置参数

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        线性层的前向传播。

        参数:
            x (torch.Tensor): 输入张量。

        返回:
            torch.Tensor: 经过线性变换的张量。

        说明:
            - 调用 linear 函数，自动处理量化权重和偏置项的计算。
        """
        return linear(x, self.weight, self.bias)  # 调用linear函数执行线性变换


In [ ]:
class ColumnParallelLinear(Linear):
    """
    列并行线性层（Column Parallel Linear），将输出特征分割到多个分布式进程中。

    参数：
        in_features (int): 输入特征的数量。
        out_features (int): 总输出特征数量。
        bias (bool): 是否包含偏置项，默认为 False。
        dtype (optional): 数据类型，默认为 `torch.bfloat16`。
    """
    def __init__(self, in_features: int, out_features: int, bias: bool = False, dtype = None):
        # 确保总输出特征数量可以被世界大小整除，以实现均匀分割
        assert out_features % world_size == 0, f"输出特征数必须能被 world_size 整除 (world_size={world_size})"

        # 计算当前进程负责的部分输出特征数
        self.part_out_features = out_features // world_size  # Calculate output features per process (计算每个进程的输出特征数)

        # 调用父类 Linear 的初始化，创建一个 in_features 到 part_out_features 的线性层
        super().__init__(in_features, self.part_out_features, bias, dtype)  # Initialize parent class with partial output features (使用部分输出特征初始化父类)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        列并行线性层的前向传播。

        参数：
            x (torch.Tensor): 输入张量。

        返回：
            torch.Tensor: 经过线性变换后的张量，进行列并行计算。
        """
        # 进行线性变换
        y = linear(x, self.weight, self.bias)  # Perform linear transformation on input tensor (对输入张量进行线性变换)
        return y


class RowParallelLinear(Linear):
    """
    行并行线性层（Row Parallel Linear），将输入特征分割到多个分布式进程中。

    参数：
        in_features (int): 总输入特征数量。
        out_features (int): 输出特征的数量。
        bias (bool): 是否包含偏置项，默认为 False。
        dtype (optional): 数据类型，默认为 `torch.bfloat16`。
    """
    def __init__(self, in_features: int, out_features: int, bias: bool = False, dtype = None):
        # 确保输入特征数量可以被世界大小整除，以实现均匀分割
        assert in_features % world_size == 0, f"输入特征数必须能被 world_size 整除 (world_size={world_size})"

        # 计算当前进程负责的部分输入特征数
        self.part_in_features = in_features // world_size  # Calculate input features per process (计算每个进程的输入特征数)

        # 调用父类 Linear 的初始化，创建一个 part_in_features 到 out_features 的线性层
        super().__init__(self.part_in_features, out_features, bias, dtype)  # Initialize parent class with partial input features (使用部分输入特征初始化父类)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        行并行线性层的前向传播。

        参数：
            x (torch.Tensor): 输入张量。

        返回：
            torch.Tensor: 经过线性变换后的张量，进行行并行计算。
        """
        # 进行线性变换
        y = linear(x, self.weight)  # Perform linear transformation without bias (执行不带偏置的线性变换)

        # 如果是分布式环境（world_size > 1），则对 y 进行 all_reduce 操作，使所有进程的计算结果进行累加
        if world_size > 1:  # If in distributed environment (如果在分布式环境中)
            dist.all_reduce(y)  # Sum up results across all processes (对所有进程的结果进行求和)

        # 如果存在偏置项，则加上偏置
        if self.bias is not None:  # If bias exists (如果存在偏置)
            y += self.bias  # Add bias to output (将偏置加到输出上)

        return y


class RMSNorm(nn.Module):
    """
    均方根归一化（RMSNorm），用于对输入张量进行归一化。

    该方法不同于标准 LayerNorm，不依赖均值，而是基于均方根（RMS）进行归一化。

    参数：
        dim (int): 输入张量的维度。
        eps (float): 用于数值稳定性的 epsilon 值，默认为 1e-6。
    """
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()  # Initialize parent class (初始化父类)
        self.dim = dim  # 记录输入维度
        self.eps = eps  # 记录 epsilon 值

        # 归一化的可训练缩放参数，初始化为全 1
        self.weight = nn.Parameter(torch.ones(dim))  # Create trainable scale parameter initialized as ones (创建初始化为1的可训练缩放参数)

    def forward(self, x: torch.Tensor):
        """
        均方根归一化的前向传播。

        参数：
            x (torch.Tensor): 输入张量。

        返回：
            torch.Tensor: 归一化后的张量，保持输入形状不变。
        """
        # 使用 torch 的 rms_norm 进行均方根归一化
        return F.rms_norm(x, (self.dim,), self.weight, self.eps)  # Apply RMS normalization using torch's function (使用torch的函数应用RMS归一化)


In [ ]:
import torch  # 导入PyTorch主库 (Import PyTorch main library)
import torch.nn as nn  # 导入神经网络模块 (Import neural network module)
import math  # 导入数学运算库 (Import math library)
from typing import Optional  # 导入类型提示 (Import type hints)

# 预计算旋转位置编码的频率值
# 该函数用于计算基于旋转位置编码的复指数值
# 主要目的是为了加速计算，避免在每次前向传播时重新计算这些值
def precompute_freqs_cis(args: ModelArgs) -> torch.Tensor:
    """
    预计算旋转位置编码的频率值。

    参数：
        args (ModelArgs): 包含位置编码参数的模型参数。

    返回：
        torch.Tensor: 预计算的复指数值，用于旋转位置编码。
    """
    dim = args.qk_rope_head_dim  # 旋转位置编码的维度,是64
    seqlen = args.max_seq_len  # 最大序列长度
    beta_fast = args.beta_fast  # 快速调整参数
    beta_slow = args.beta_slow  # 缓慢调整参数
    base = args.rope_theta  # 旋转位置编码的基数
    factor = args.rope_factor  # 旋转位置编码的缩放因子

    # 计算旋转位置编码修正维度
    def find_correction_dim(num_rotations, dim, base, max_seq_len):
        """
        计算旋转位置编码的修正维度。
        """
        return dim * math.log(max_seq_len / (num_rotations * 2 * math.pi)) / (2 * math.log(base))  # 使用对数计算修正维度 (Calculate correction dimension using logarithm)

    # 计算旋转位置编码修正范围
    def find_correction_range(low_rot, high_rot, dim, base, max_seq_len):
        """
        计算旋转位置编码修正范围。
        参数：
            low_rot (int): 低频旋转数量。
            high_rot (int): 高频旋转数量。
            dim (int): 旋转位置编码的维度。
            base (float): 旋转位置编码的基数。
            max_seq_len (int): 最大序列长度。
        返回：
            tuple: 包含修正范围的元组，第一个元素为下界，第二个元素为上界。
        """
        low = math.floor(find_correction_dim(low_rot, dim, base, max_seq_len))  # 计算下界并向下取整 (Calculate lower bound and round down)
        high = math.ceil(find_correction_dim(high_rot, dim, base, max_seq_len))  # 计算上界并向上取整 (Calculate upper bound and round up)
        return max(low, 0), min(high, dim-1)  # 确保范围在有效区间内 (Ensure range is within valid interval)

    # 计算线性斜坡因子，用于平滑过渡
    def linear_ramp_factor(min, max, dim):
        """
        计算线性斜坡因子。
        参数：
            min (int): 最小值。
            max (int): 最大值。
            dim (int): 维度。
        返回：
            torch.Tensor: 线性斜坡因子。
        """
        if min == max:  # 如果最小值等于最大值，则稍微增加最大值 (If min equals max, slightly increase max)
            max += 0.001
        linear_func = (torch.arange(dim, dtype=torch.float32) - min) / (max - min)  # 计算线性函数 (Calculate linear function)
        ramp_func = torch.clamp(linear_func, 0, 1)  # 将值限制在[0,1]范围内 (Clamp values to [0,1] range)
        return ramp_func

    # 计算基础频率
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))  # 计算基础频率序列 (Calculate base frequency sequence)

    # 如果序列长度超过原始最大长度，则进行修正
    if seqlen > args.original_seq_len:  # 检查是否需要序列长度修正 (Check if sequence length correction is needed)
        low, high = find_correction_range(beta_fast, beta_slow, dim, base, args.original_seq_len)  # 计算修正范围 (Calculate correction range)
        smooth = 1 - linear_ramp_factor(low, high, dim // 2)  # 计算平滑因子 (Calculate smoothing factor)
        freqs = freqs / factor * (1 - smooth) + freqs * smooth  # 应用平滑修正 (Apply smooth correction)

    t = torch.arange(seqlen)  # 创建位置索引序列 (Create position index sequence)
    freqs = torch.outer(t, freqs)  # 计算外积得到频率矩阵 (Calculate outer product to get frequency matrix)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # 将频率转换为复数形式 (Convert frequencies to complex form)
    return freqs_cis

In [ ]:
args = ModelArgs()  # 初始化模型参数
precompute_freqs_cis(args).shape#(1024, 64)

torch.Size([16384, 32])

In [ ]:


# 应用旋转位置编码到输入张量
# 该函数使用预计算的复指数值对输入进行旋转编码

def apply_rotary_emb(x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
    """
    应用旋转位置编码到输入张量。

    参数：
        x (torch.Tensor): 输入张量。
        freqs_cis (torch.Tensor): 预计算的复指数值。

    返回：
        torch.Tensor: 旋转编码后的张量。
    """
    dtype = x.dtype  # 保存原始数据类型 (Save original data type)
    print(f'apply_rotary_emb 开始{x.shape}')
    x = torch.view_as_complex(x.float().view(*x.shape[:-1], -1, 2))  # 将张量重塑并转换为复数形式 (Reshape tensor and convert to complex form)
    print(f'apply_rotary_emb view_as_complex后{x.shape}')
    freqs_cis = freqs_cis.view(1, x.size(1), 1, x.size(-1))  # 调整频率张量的形状以匹配输入 (Adjust frequency tensor shape to match input)
    print(f'apply_rotary_emb freqs_cis.view{freqs_cis.shape}')
    y = torch.view_as_real(x * freqs_cis).flatten(3)  # 执行复数乘法并转回实数形式 (Perform complex multiplication and convert back to real)
    print(f'apply_rotary_emb y{y.shape}')
    return y.to(dtype)  # 转换回原始数据类型 (Convert back to original data type)

# 多头注意力层（MLA）  多头隐性注意力
# 该类实现了标准的多头注意力机制，并结合了旋转位置编码
class MLA(nn.Module):
    """
    多头注意力层（MLA）。

    属性:
        dim (int): 输入特征的维度。
        n_heads (int): 注意力头的数量。
        n_local_heads (int): 分布式系统中用于局部注意力的头数量。
        q_lora_rank (int): 查询低秩投影的秩。
        kv_lora_rank (int): 键值低秩投影的秩。
        qk_nope_head_dim (int): 非位置查询/键投影的维度。
        qk_rope_head_dim (int): 旋转位置查询/键投影的维度。
        qk_head_dim (int): 查询/键投影的总维度。
        v_head_dim (int): 值投影的维度。
        softmax_scale (float): 注意力计算中Softmax的缩放因子。
    """
    def __init__(self, args: ModelArgs):
        super().__init__()
        # 初始化各个参数
        self.dim = args.dim  # 输入的特征维度 (Input feature dimension)
        self.n_heads = args.n_heads  # 注意力头的数量 (Number of attention heads)
        self.n_local_heads = args.n_heads // world_size  # 分布式环境中的局部注意力头数 (Local attention heads in distributed environment)
        self.q_lora_rank = args.q_lora_rank  # 查询低秩投影的秩 (Rank for query low-rank projection)
        self.kv_lora_rank = args.kv_lora_rank  # 键值低秩投影的秩 (Rank for key-value low-rank projection)
        self.qk_nope_head_dim = args.qk_nope_head_dim  # 非位置查询/键的维度 (Dimension for non-positional query/key)
        self.qk_rope_head_dim = args.qk_rope_head_dim  # 旋转位置查询/键的维度 (Dimension for rotary positional query/key)
        self.qk_head_dim = args.qk_nope_head_dim + args.qk_rope_head_dim  # 查询/键投影的总维度 (Total dimension for query/key projection)
        self.v_head_dim = args.v_head_dim  # 值投影的维度 (Dimension for value projection)

        # 如果q_lora_rank为0，直接使用列并行线性层，否则使用低秩投影和标准化
        if self.q_lora_rank == 0:
            self.wq = ColumnParallelLinear(self.dim, self.n_heads * self.qk_head_dim)  # 直接使用列并行线性层 (Use column parallel linear layer directly)
        else:
            self.wq_a = Linear(self.dim, self.q_lora_rank)  # 低秩投影的第一部分 (First part of low-rank projection)
            self.q_norm = RMSNorm(self.q_lora_rank)  # 低秩投影的标准化 (Normalization for low-rank projection)
            self.wq_b = ColumnParallelLinear(self.q_lora_rank, self.n_heads * self.qk_head_dim)  # 低秩投影的第二部分 (Second part of low-rank projection)

        # 键值投影和标准化
        self.wkv_a = Linear(self.dim, self.kv_lora_rank + self.qk_rope_head_dim)  # 键值低秩投影 (Key-value low-rank projection)
        self.kv_norm = RMSNorm(self.kv_lora_rank)  # 键值的标准化 (Normalization for key-value)
        self.wkv_b = ColumnParallelLinear(self.kv_lora_rank, self.n_heads * (self.qk_nope_head_dim + self.v_head_dim))  # 键值投影 (Key-value projection)
        self.wo = RowParallelLinear(self.n_heads * self.v_head_dim, self.dim)  # 输出投影 (Output projection)
        self.softmax_scale = self.qk_head_dim ** -0.5  # Softmax缩放因子 (Softmax scaling factor)

        # 如果最大序列长度大于原始序列长度，调整softmax_scale
        if args.max_seq_len > args.original_seq_len:
            mscale = 0.1 * args.mscale * math.log(args.rope_factor) + 1.0  # 计算缩放因子 (Calculate scaling factor)
            self.softmax_scale = self.softmax_scale * mscale * mscale  # 应用缩放 (Apply scaling)

        # 根据注意力实现类型选择不同的缓存方式
        if attn_impl == "naive":
            # 在"naive"实现下缓存k和v
            self.register_buffer("k_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.n_local_heads, self.qk_head_dim), persistent=False)  # 键缓存 (Key cache)
            self.register_buffer("v_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.n_local_heads, self.v_head_dim), persistent=False)  # 值缓存 (Value cache)
        else:
            # 在其他实现下缓存kv和pe
            self.register_buffer("kv_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.kv_lora_rank), persistent=False)  # 键值缓存 (Key-value cache)
            self.register_buffer("pe_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.qk_rope_head_dim), persistent=False)  # 位置编码缓存 (Position encoding cache)

    def forward(self, x: torch.Tensor, start_pos: int, freqs_cis: torch.Tensor, mask: Optional[torch.Tensor]):
        """
        多头注意力层的前向传播。

        参数:
            x (torch.Tensor): 输入张量，形状为(batch_size, seq_len, dim)。
            start_pos (int): 缓存的起始位置。
            freqs_cis (torch.Tensor): 预计算的旋转嵌入的复数指数值。
            mask (Optional[torch.Tensor]): 掩码张量，用于排除某些位置的注意力计算。

        返回:
            torch.Tensor: 输出张量，形状与输入相同。
        """
        bsz, seqlen, _ = x.size()  # 获取输入张量的维度 (Get input tensor dimensions)
        end_pos = start_pos + seqlen  # 计算结束位置 (Calculate end position)

        # 计算查询（q）
        if self.q_lora_rank == 0:
            q = self.wq(x)  # 直接计算查询 (Calculate query directly)
        else:
            q = self.wq_b(self.q_norm(self.wq_a(x)))  # 使用低秩投影计算查询 (Calculate query using low-rank projection)

        # 重塑查询张量的形状
        q = q.view(bsz, seqlen, self.n_local_heads, self.qk_head_dim)  # 重塑查询张量 (Reshape query tensor)
        print(f"MLA q shape: {q.shape}")
        
        # 分割查询张量为非位置部分和位置编码部分
        q_nope, q_pe = torch.split(q, [self.qk_nope_head_dim, self.qk_rope_head_dim], dim=-1)  # 分割查询张量 (Split query tensor)
        print(f"q_nope shape: {q_nope.shape}, q_pe shape: {q_pe.shape}")
        q_pe = apply_rotary_emb(q_pe, freqs_cis)  # 应用旋转位置编码 (Apply rotary position encoding)

        # 计算键值（kv）
        kv = self.wkv_a(x)  # 计算键值 (Calculate key-value)
        print("kv shape:", kv.shape)
        kv, k_pe = torch.split(kv, [self.kv_lora_rank, self.qk_rope_head_dim], dim=-1)  # 分割键值张量 (Split key-value tensor)
        print(f"kv shape: {kv.shape}, k_pe shape: {k_pe.shape}")
        k_pe = apply_rotary_emb(k_pe.unsqueeze(2), freqs_cis)  # 应用旋转位置编码到键 (Apply rotary position encoding to key)

        # 判断是否使用"naive"注意力实现
        if attn_impl == "naive":
            q = torch.cat([q_nope, q_pe], dim=-1)  # 拼接查询张量 (Concatenate query tensor)
            kv = self.wkv_b(self.kv_norm(kv))  # 计算键值 (Calculate key-value)
            kv = kv.view(bsz, seqlen, self.n_local_heads, self.qk_nope_head_dim + self.v_head_dim)  # 重塑键值张量 (Reshape key-value tensor)
            k_nope, v = torch.split(kv, [self.qk_nope_head_dim, self.v_head_dim], dim=-1)  # 分割键值 (Split key-value)
            print(f'MLA k_pe.expand {k_pe.expand(-1, -1, self.n_local_heads, -1).shape}')
            k = torch.cat([k_nope, k_pe.expand(-1, -1, self.n_local_heads, -1)], dim=-1)  # 拼接键张量 (Concatenate key tensor)
            print(f"k shape: {k.shape}")
            self.k_cache[:bsz, start_pos:end_pos] = k  # 更新键缓存 (Update key cache)
            self.v_cache[:bsz, start_pos:end_pos] = v  # 更新值缓存 (Update value cache)
            scores = torch.einsum("bshd,bthd->bsht", q, self.k_cache[:bsz, :end_pos]) * self.softmax_scale  # 计算注意力得分 (Calculate attention scores)
        else:
            wkv_b = self.wkv_b.weight if self.wkv_b.scale is None else weight_dequant(self.wkv_b.weight, self.wkv_b.scale, block_size)  # 获取权重 (Get weights)
            wkv_b = wkv_b.view(self.n_local_heads, -1, self.kv_lora_rank)  # 重塑权重 (Reshape weights)
            print(f'wkv_b shape{wkv_b.shape}')
            q_nope = torch.einsum("bshd,hdc->bshc", q_nope, wkv_b[:, :self.qk_nope_head_dim])  # 计算非位置查询 (Calculate non-positional query)
            self.kv_cache[:bsz, start_pos:end_pos] = self.kv_norm(kv)  # 更新键值缓存 (Update key-value cache)
            self.pe_cache[:bsz, start_pos:end_pos] = k_pe.squeeze(2)  # 更新位置编码缓存 (Update position encoding cache)
            scores = (torch.einsum("bshc,btc->bsht", q_nope, self.kv_cache[:bsz, :end_pos]) +
                      torch.einsum("bshr,btr->bsht", q_pe, self.pe_cache[:bsz, :end_pos])) * self.softmax_scale  # 计算注意力得分 (Calculate attention scores)

        print(f'scores shape{scores.shape}')
        # 如果有mask，添加到得分上
        if mask is not None:
            scores += mask.unsqueeze(1)  # 应用注意力掩码 (Apply attention mask)

        # 计算softmax得分
        scores = scores.softmax(dim=-1, dtype=torch.float32).type_as(x)  # 计算softmax (Calculate softmax)

        # 根据注意力实现类型，选择不同的计算方式
        if attn_impl == "naive":
            x = torch.einsum("bsht,bthd->bshd", scores, self.v_cache[:bsz, :end_pos])  # 计算注意力输出 (Calculate attention output)
        else:
            x = torch.einsum("bsht,btc->bshc", scores, self.kv_cache[:bsz, :end_pos])  # 计算中间结果 (Calculate intermediate result)
            x = torch.einsum("bshc,hdc->bshd", x, wkv_b[:, -self.v_head_dim:])  # 计算最终输出 (Calculate final output)
        print(f'多头注意力后x shape{x.shape}')
        
        x = self.wo(x.flatten(2))  # 通过输出投影层 (Through output projection layer)
        return x



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple

class MLP(nn.Module):
    """
    多层感知机（MLP），用于前馈计算。

    该模块包含三个线性变换层，分别是 w1、w2 和 w3，用于特征变换和计算。

    属性:
        w1 (nn.Module): 线性层，用于从输入层到隐藏层的转换。
        w2 (nn.Module): 线性层，用于从隐藏层到输出层的转换。
        w3 (nn.Module): 额外的线性层，用于特征变换。
    """
    def __init__(self, dim: int, inter_dim: int):
        """
        初始化 MLP 层。

        参数:
            dim (int): 输入和输出的维度（维度保持一致）。
            inter_dim (int): 隐藏层的维度。
        """
        super().__init__()
        self.w1 = ColumnParallelLinear(dim, inter_dim)  # 第一层线性变换
        self.w2 = RowParallelLinear(inter_dim, dim)     # 第二层线性变换（回到原始维度）
        self.w3 = ColumnParallelLinear(dim, inter_dim)  # 额外的线性变换层

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        MLP 的前向计算。

        参数:
            x (torch.Tensor): 输入张量，形状为 (batch_size, dim)。

        返回:
            torch.Tensor: 经过 MLP 计算后的输出张量，形状为 (batch_size, dim)。
        """
        return self.w2(F.silu(self.w1(x)) * self.w3(x))  # 使用 SiLU 激活函数进行非线性变换，并结合 w3(x) 进行特征变换


class Gate(nn.Module):
    """
    用于 Mixture-of-Experts（MoE）模型的门控机制（Gating Mechanism）。

    该模块用于在多个专家（Expert）之间进行路由选择，决定每个输入数据应该被送到哪些专家进行计算。

    属性:
        dim (int): 输入特征的维度。
        topk (int): 每个输入激活的专家数（选择 top-k 个专家）。
        n_groups (int): 专家被划分的组数（用于路由分组）。
        topk_groups (int): 每个输入路由到的专家组数。
        score_func (str): 计算分数的函数（可选 "softmax" 或 "sigmoid"）。
        route_scale (float): 路由权重的缩放因子。
        weight (torch.nn.Parameter): 可训练参数，表示门控网络的权重矩阵。
        bias (Optional[torch.nn.Parameter]): 可选的偏置项，仅当输入维度为 7168 时存在。
    """
    def __init__(self, args: ModelArgs):
        """
        初始化 Gate 模块。

        参数:
            args (ModelArgs): 传入的模型参数对象，包含 MoE 相关的超参数。
        返回：
            None
        """
        super().__init__()
        self.dim = args.dim  # 输入特征的维度
        self.topk = args.n_activated_experts  # 选择的前 top-k 个专家
        self.n_groups = args.n_expert_groups  # 专家分组的数量
        self.topk_groups = args.n_limited_groups  # 选择的前 top-k 组
        self.score_func = args.score_func  # 计算分数的方式
        self.route_scale = args.route_scale  # 路由权重的缩放比例

        # 可训练权重参数（用于计算门控分数）
        self.weight = nn.Parameter(torch.empty(args.n_routed_experts, args.dim))

        # 只有当 dim = 7168 时，才会添加可训练的偏置项
        self.bias = nn.Parameter(torch.empty(args.n_routed_experts)) if self.dim == 7168 else None

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        计算门控权重，并确定选择哪些专家进行计算。

        参数:
            x (torch.Tensor): 输入张量，形状为 (batch_size, dim)。

        返回:
            Tuple[torch.Tensor, torch.Tensor]:
                - 选择的专家权重 (batch_size, topk)
                - 选择的专家索引 (batch_size, topk)
        """
        # 一行代码f格式打印x的形状
        print(f"Gate之前 x shape: {x.shape}")
        scores = linear(x, self.weight)  # 计算输入与门控权重的线性变换

        # 根据 score_func 选择 softmax 或 sigmoid 进行归一化
        if self.score_func == "softmax":
            scores = scores.softmax(dim=-1, dtype=torch.float32)
        else:
            scores = scores.sigmoid()
        print(f'softmax后各专家的分数shape {scores.shape}') #（256,16） 把每一个token给16个专家都计算一个分数
        original_scores = scores  # 保存原始分数

        # 若存在偏置项，则加上偏置
        if self.bias is not None:
            scores = scores + self.bias

        # 若使用多个专家组，则进行分组处理
        if self.n_groups > 1:
            scores = scores.view(x.size(0), self.n_groups, -1)  # 重新 reshape 为 (batch_size, n_groups, 每组的专家数)

            # 计算每组的得分，若无偏置，则取最大值；若有偏置，则取 top-2 得分之和
            if self.bias is None:
                group_scores = scores.amax(dim=-1)
            else:
                group_scores = scores.topk(2, dim=-1)[0].sum(dim=-1)

            # 选择得分最高的 topk_groups 组，并生成掩码
            indices = group_scores.topk(self.topk_groups, dim=-1)[1]
            mask = torch.zeros_like(scores[..., 0]).scatter_(1, indices, True)
            scores = (scores * mask.unsqueeze(-1)).flatten(1)  # 仅保留选中的专家分数

        # 选择得分最高的 top-k 个专家
        indices = torch.topk(scores, self.topk, dim=-1)[1]
        #打印indices
        print("Gate indices shape:", indices.shape)
        print("Gate indices value:", indices) #因为没有没训练，所以看到的token选择的专家是一样的
        # 计算最终的专家权重
        weights = original_scores.gather(1, indices)
        #打印weights
        print("Gate weights :", weights.shape)
        # 若使用 sigmoid，则需要归一化
        if self.score_func == "sigmoid":
            weights /= weights.sum(dim=-1, keepdim=True)

        weights *= self.route_scale  # 乘以路由缩放因子
        return weights.type_as(x), indices  # 返回计算后的权重和选择的专家索引


class Expert(nn.Module):
    """
    专家（Expert）层，用于 Mixture-of-Experts（MoE）模型。

    该模块实现了一个独立的专家网络，每个专家由三层线性变换层组成。

    属性:
        w1 (nn.Module): 线性层，从输入到隐藏层的变换。
        w2 (nn.Module): 线性层，从隐藏层到输出的变换。
        w3 (nn.Module): 额外的线性层，用于特征变换。
    """
    def __init__(self, dim: int, inter_dim: int):
        """
        初始化 Expert 层。

        参数:
            dim (int): 输入和输出的维度。
            inter_dim (int): 隐藏层的维度。
        """
        super().__init__()
        self.w1 = nn.Linear(dim, inter_dim)  # 输入到隐藏层的线性变换
        self.w2 = nn.Linear(inter_dim, dim)  # 隐藏层到输出层的线性变换
        self.w3 = nn.Linear(dim, inter_dim)  # 额外的线性变换层

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Expert 层的前向计算。

        参数:
            x (torch.Tensor): 输入张量，形状为 (batch_size, dim)。

        返回:
            torch.Tensor: 经过 Expert 计算后的输出张量，形状为 (batch_size, dim)。
        """
        return self.w2(F.silu(self.w1(x)) * self.w3(x))  # 先经过 w1 进行非线性变换，再与 w3 计算的结果相乘，最后通过 w2 输出


In [ ]:
import torch  # 导入PyTorch主包
import torch.nn as nn  # 导入神经网络模块
import torch.distributed as dist  # 导入分布式训练模块
from typing import Optional  # 导入类型提示模块

class MoE(nn.Module):
    """
    MoE（Mixture-of-Experts，专家混合）模块，用于选择性地激活多个专家网络，提高计算效率。

    主要属性：
        dim (int): 输入特征的维度。
        n_routed_experts (int): 该模型中的总专家数量。
        n_local_experts (int): 在分布式环境中，每个设备处理的专家数量。
        n_activated_experts (int): 每个输入激活的专家数量。
        gate (nn.Module): 用于计算输入到各专家的分配权重的门控机制。
        experts (nn.ModuleList): 专家网络列表，每个专家都是一个神经网络模块。
        shared_experts (nn.Module): 共享专家网络，对所有输入均生效。
    """
    def __init__(self, args: ModelArgs):
        """
        初始化 MoE 模块。

        参数：
            args (ModelArgs): 包含 MoE 参数的模型配置。
        """
        super().__init__()  # 调用父类初始化
        self.dim = args.dim  # 设置输入特征维度

        # 确保专家数量可以被世界大小整除（用于分布式训练）。
        assert args.n_routed_experts % world_size == 0, f"专家数量必须被 world_size 整除 (world_size={world_size})"

        self.n_routed_experts = args.n_routed_experts  # 设置总专家数量
        self.n_local_experts = args.n_routed_experts // world_size  # 计算每个设备的专家数量
        self.n_activated_experts = args.n_activated_experts #6个

        # 计算当前设备负责的专家索引范围。
        self.experts_start_idx = rank * self.n_local_experts  # 计算当前设备专家的起始索引
        self.experts_end_idx = self.experts_start_idx + self.n_local_experts  # 计算当前设备专家的结束索引

        # 门控机制，用于决定输入数据分配给哪些专家。
        self.gate = Gate(args)  # 初始化门控机制

        # 仅在当前设备上初始化其负责的专家，其余设为 None。，16个专家
        self.experts = nn.ModuleList([Expert(args.dim, args.moe_inter_dim) if self.experts_start_idx <= i < self.experts_end_idx
                                      else None
                                      for i in range(self.n_routed_experts)])  # 初始化专家列表

        # 共享专家，对所有输入均适用，每次都会走的专家
        self.shared_experts = MLP(args.dim, args.n_shared_experts * args.moe_inter_dim)  # 初始化共享专家网络

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        MoE 模块的前向传播。

        参数：
            x (torch.Tensor): 输入张量。

        返回：
            torch.Tensor: 经过专家计算后的输出张量。
        """
        shape = x.size()  # 保存输入张量的原始形状
        #打印x的shape
        print(f"MoE x shape: {x.shape}")  # 打印输入张量的形状
        x = x.view(-1, self.dim) #把x的bs和seq_len展平了

        # 通过门控机制获取专家索引及其权重。
        weights, indices = self.gate(x)  # 获取每个输入应该由哪些专家处理及其权重
        y = torch.zeros_like(x) #y是和x一样尺寸（256,2048）的，但是里边的值都是0

        # 统计各专家被选中的次数。
        counts = torch.bincount(indices.flatten(), minlength=self.n_routed_experts).tolist()  # 计算每个专家被选中的次数

        # 遍历当前设备管理的专家，并处理其对应的输入。
        for i in range(self.experts_start_idx, self.experts_end_idx):  # 遍历当前设备的专家
            if counts[i] == 0:  # 如果该专家未被选中
                continue  # 如果当前专家未被选中，则跳过。
            expert = self.experts[i]  # 获取当前专家
            idx, top = torch.where(indices == i) #indices尺寸（256,6），这句操作就把256个token中，谁选择了0号专家，就把它筛选
            print(f'moe-idx{idx}')  # 打印选择当前专家的输入索引
            y[idx] += expert(x[idx]) * weights[idx, top, None] #把选择了0号专家的token喂给0号专家，乘以0号专家的权重

        # 共享专家计算。
        z = self.shared_experts(x)  # 所有输入都通过共享专家计算

        # 若为多设备环境，则进行全局同步。
        if world_size > 1:  # 如果是分布式环境
            dist.all_reduce(y)  # 同步所有设备的计算结果
        #打印，y，z的shape
        print(f"MoE y shape: {y.shape}, MoE z shape: {z.shape}")  # 打印输出张量的形状
        return (y + z).view(shape) #又变为x最初的shape




import torch  # 导入PyTorch主包
import torch.nn as nn  # 导入神经网络模块
import torch.distributed as dist  # 导入分布式训练模块
from typing import Optional  # 导入类型提示模块

class Block(nn.Module):
    """
    Transformer 块，结合了注意力机制和前馈网络。

    属性:
        attn (nn.Module): 多头注意力（MLA, Multi-Head Attention）。
        ffn (nn.Module): 前馈神经网络（MLP 或 MoE）。
        attn_norm (nn.Module): 注意力层的归一化层。
        ffn_norm (nn.Module): 前馈网络的归一化层。
    """
    def __init__(self, layer_id: int, args: ModelArgs):
        """
        初始化 Transformer 块。

        参数:
            layer_id (int): 该层在 Transformer 中的索引。
            args (ModelArgs): 包含 Transformer 相关参数的配置对象。
        """
        super().__init__()  # 调用父类初始化
        self.attn = MLA(args)  # 多头潜在注意力机制
        # 如果 layer_id 小于稠密层数量，则使用 MLP，否则使用 MoE 结构
        self.ffn = MLP(args.dim, args.inter_dim) if layer_id < args.n_dense_layers else MoE(args)  # 根据层位置选择前馈网络类型
        self.attn_norm = RMSNorm(args.dim)  # 注意力归一化层
        self.ffn_norm = RMSNorm(args.dim)   # 前馈归一化层

    def forward(self, x: torch.Tensor, start_pos: int, freqs_cis: torch.Tensor, mask: Optional[torch.Tensor]) -> torch.Tensor:
        """
        Transformer 块的前向传播。

        参数:
            x (torch.Tensor): 输入张量。
            start_pos (int): 序列的起始位置。
            freqs_cis (torch.Tensor): 预计算的旋转嵌入复指数值。
            mask (Optional[torch.Tensor]): 掩码张量，用于排除特定位置的注意力。

        返回:
            torch.Tensor: 经过 Transformer 块计算后的输出张量。
        """
        x = x + self.attn(self.attn_norm(x), start_pos, freqs_cis, mask)  # 归一化后进行注意力计算并残差连接
        x = x + self.ffn(self.ffn_norm(x))  # 归一化后进入前馈网络并残差连接
        return x


class Transformer(nn.Module):
    """
    Transformer 模型，包括嵌入层、多层 Transformer 块、最终归一化层和输出层。

    属性:
        max_seq_len (int): 最大序列长度。
        embed (nn.Module): 词嵌入层。
        layers (torch.nn.ModuleList): Transformer 块的列表。
        norm (nn.Module): 所有 Transformer 层之后的归一化层。
        head (nn.Module): 输出投影层，将隐藏状态映射到词汇表大小。
        freqs_cis (torch.Tensor): 预计算的旋转嵌入复指数值。
    """
    def __init__(self, args: ModelArgs):
        """
        初始化 Transformer 模型。

        参数:
            args (ModelArgs): 包含 Transformer 相关参数的配置对象。
        """
        global world_size, rank  # 声明全局变量
        world_size = dist.get_world_size() if dist.is_initialized() else 1  # 获取分布式训练的总进程数
        rank = dist.get_rank() if dist.is_initialized() else 0  # 获取当前进程的 rank 值
        Linear.dtype = torch.float8_e4m3fn if args.dtype == "fp8" else torch.bfloat16  # 设置默认数据类型
        super().__init__()  # 调用父类初始化
        self.max_seq_len = args.max_seq_len  # 最大序列长度
        self.embed = ParallelEmbedding(args.vocab_size, args.dim)  # 词嵌入层，支持并行计算
        self.layers = torch.nn.ModuleList()  # 初始化Transformer块列表
        for layer_id in range(args.n_layers):  # 循环创建Transformer块
            self.layers.append(Block(layer_id, args))  # 添加多个 Transformer 块
        self.norm = RMSNorm(args.dim)  # 归一化层
        self.head = ColumnParallelLinear(args.dim, args.vocab_size, dtype=torch.get_default_dtype())  # 输出投影层
        self.register_buffer("freqs_cis", precompute_freqs_cis(args), persistent=False)  # 预计算旋转位置编码

    @torch.inference_mode()  # 推理模式装饰器，禁用梯度计算
    def forward(self, tokens: torch.Tensor, start_pos: int = 0):
        """
        Transformer 的前向传播。

        参数:
            tokens (torch.Tensor): 形状为 (batch_size, seq_len) 的输入 token ID。
            start_pos (int, 可选): 序列的起始位置，默认为 0。

        返回:
            torch.Tensor: 形状为 (batch_size, vocab_size) 的 logits。
        """
        seqlen = tokens.size(1)  # 获取输入序列长度
        h = self.embed(tokens)  # 通过词嵌入层获取 token 表示
        freqs_cis = self.freqs_cis[start_pos:start_pos+seqlen]  # 获取对应位置的旋转位置编码
        mask = None  # 初始化掩码为None
        if seqlen > 1:  # 如果序列长度大于1
            mask = torch.full((seqlen, seqlen), float("-inf"), device=tokens.device).triu_(1)  # 构造上三角掩码（防止未来信息泄露）
        for layer in self.layers:  # 遍历所有Transformer块
            h = layer(h, start_pos, freqs_cis, mask)  # 依次通过每个 Transformer 块
        h = self.norm(h)[:, -1]  # 归一化后取最后一个时间步的输出
        logits = self.head(h)  # 通过输出投影层计算 logits
        if world_size > 1:  # 如果是分布式环境
            all_logits = [torch.empty_like(logits) for _ in range(world_size)]  # 创建存储所有进程logits的列表
            dist.all_gather(all_logits, logits)  # 在所有进程间收集 logits
            logits = torch.cat(all_logits, dim=-1)  # 拼接所有进程的 logits
        return logits


if __name__ == "__main__":  # 如果是主程序
    torch.set_default_dtype(torch.bfloat16)  # 设置默认数据类型
    torch.set_default_device("cuda")  # 设置默认计算设备为 GPU
    torch.manual_seed(0)  # 设置随机种子，保证可复现性
    args = ModelArgs()  # 初始化模型参数
    x = torch.randint(0, args.vocab_size, (2, 128))  # 2个样本，随机生成 token ID 作为输入
    print(f'输入x {x}')  # 打印输入数据
    model = Transformer(args)  # 初始化 Transformer 模型
    print(model(x).size())  # 运行模型并打印输出张量的形状，为啥只有一个词，因为只取了最后一个的输出，可以理解这个样例像咱们的bert基座一样，让大家去练习一个分类问题
    print('-'*50)  # 打印分隔线
    print(model)  # 打印模型结构


输入x tensor([[ 22741,   9697,  67191,  80572, 102153,  33804,  97703,   7133,  79442,
          88633,  40990,   5717,  99598,  15084,  76478,  96608,  93964,   9011,
          82122,  44591,  36312,  41457,    451,  29075,  22534,  69937,  78376,
          51202,  63807,  16739,  71190,  98135,  41314,  31347,  46940,   1366,
          76072,   4177,  88254,  98414,  64326,  90258,  98531,  84276,  55948,
          42091,  95705,  34653,  56968,  81333,  92365,  47543,  96507, 102165,
          31402,  50662,  44457,  85499,   7149, 102175,  41399,  14557,   1525,
          98829,  45593,  51562,  19597,  74482,  97871,  15389,  55267,  32132,
          73288,  85122,  58165,  46965,  98694,  73189,  56220,  12451,  68347,
         101519,   2903,  85123,  11353,  80649,  54205,  96576,  28563,  59464,
          77530,  63623,  26066,  50972,  77261,  98883,  83778,  13584,  63728,
          13459,  84498,  40664,  37703,  96879,  11094,  33929,   4116,  89902,
          40499,  90162,